# G1 Data Cleaning

This notebook supplements `report-generation_local-llm.ipynb`. It prepares the cleaned MAS G1 workbook used by the local Hugging Face report-generation workflow, so the reporting notebook can focus on KPI calculation, controlled prompting, LLM-generated commentary, and Word report production.

The notebook collects the G1 worksheet from each annual MAS form workbook, extracts the operating results by line of business, cleans the rows and numeric columns, and exports each insurer fund to a separate worksheet in one Excel file. Keeping this preprocessing step separate makes the reporting workflow easier to review, rerun, and govern.


## 1. Import packages and define paths

`pathlib.Path` keeps file paths readable across Windows and other operating systems. The input folder is set to `Data/MAS Form` because that is where the raw MAS form workbooks are stored. The output workbook is written one level up to the main `Data` folder, where the report-generation notebooks expect to find the cleaned MAS G1 data.


In [25]:
from difflib import SequenceMatcher
from pathlib import Path
import re

import pandas as pd


INPUT_DIR = Path("Data") / "MAS_Form"
OUTPUT_DIR = Path("Data")
OUTPUT_FILE = OUTPUT_DIR / "Output_MAS_G1.xlsx"
MAPPING_FILE = Path("Knowledge") / "Mapping.xlsx"

SHEET_NAME = "G1"
YEAR_PATTERNS = ("*_2022.*", "*_2023.*", "*_2024.*")

OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Find the source files

The loop below searches for all files in the input folder whose names end in the target reporting years. Sorting the list makes the workbook output order stable every time the notebook runs.


In [26]:
source_files = sorted(
    file_path
    for pattern in YEAR_PATTERNS
    for file_path in INPUT_DIR.glob(pattern)
    if file_path.is_file()
)

if not source_files:
    raise FileNotFoundError(
        f"No source files found in {INPUT_DIR.resolve()} for patterns: {YEAR_PATTERNS}"
    )

source_files


[PosixPath('Data/MAS Form/I400G_AUTO_&_GENERAL_INSURANCE_SINGAPORE_PTE._LIMITED_2022.xlsx'),
 PosixPath('Data/MAS Form/I400G_AUTO_&_GENERAL_INSURANCE_SINGAPORE_PTE._LIMITED_2023.xlsx'),
 PosixPath('Data/MAS Form/I400G_AUTO_&_GENERAL_INSURANCE_SINGAPORE_PTE._LIMITED_2024.xlsx'),
 PosixPath('Data/MAS Form/I401G_THE_WEST_OF_ENGLAND_SHIPOWNERS_MUTUAL_INSURANCE_ASSOCIATION_LUXEMBOURG_SINGAPORE_BRANCH_2022.xlsx'),
 PosixPath('Data/MAS Form/I401G_THE_WEST_OF_ENGLAND_SHIPOWNERS_MUTUAL_INSURANCE_ASSOCIATION_LUXEMBOURG_SINGAPORE_BRANCH_2023.xlsx'),
 PosixPath('Data/MAS Form/I401G_THE_WEST_OF_ENGLAND_SHIPOWNERS_MUTUAL_INSURANCE_ASSOCIATION_LUXEMBOURG_SINGAPORE_BRANCH_2024.xlsx'),
 PosixPath('Data/MAS Form/I403G_STEAMSHIP_MUTUAL_UNDERWRITING_ASSOCIATION_LIMITED_SINGAPORE_BRANCH_2022.xlsx'),
 PosixPath('Data/MAS Form/I403G_STEAMSHIP_MUTUAL_UNDERWRITING_ASSOCIATION_LIMITED_SINGAPORE_BRANCH_2023.xlsx'),
 PosixPath('Data/MAS Form/I403G_STEAMSHIP_MUTUAL_UNDERWRITING_ASSOCIATION_LIMITED_SINGAPORE_BRANCH

## 3. Define the cleaning helpers

`clean_g1_table` extracts the insurer name, Singapore Insurance Fund section, and Offshore Insurance Fund section by row and column position. It also turns the visual headings from the source sheet into `Category` and `Subcategory` columns, so the output is clearer than the merged-cell layout in the original workbook.


In [27]:
DESCRIPTION_COLUMN = "Description"

G1_OUTPUT_COLUMNS = [
    "insurer",
    "Section",
    "Category",
    "Subcategory",
    DESCRIPTION_COLUMN,
    "Row No.",
    "Cargo",
    "Marine Hull",
    "Aviation Hull",
    "Property",
    "Motor",
    "Employers' Liability",
    "Personal Accident",
    "Health",
    "Public Liability/ Product Liability",
    "Surety",
    "Engineering",
    "Professional Indemnity",
    "Credit/ Credit-related",
    "Others",
    "Liability and Others",
    "Total",
]
G1_TABLE_SECTIONS = [
    {
        "section": "Singapore Insurance Fund",
        "start_row": 14,
        "end_row": 68,
        "column_indices": list(range(2, 19)),
        "column_names": [
            DESCRIPTION_COLUMN,
            "Row No.",
            "Cargo",
            "Marine Hull",
            "Aviation Hull",
            "Property",
            "Motor",
            "Employers' Liability",
            "Personal Accident",
            "Health",
            "Public Liability/ Product Liability",
            "Surety",
            "Engineering",
            "Professional Indemnity",
            "Credit/ Credit-related",
            "Others",
            "Total",
        ],
    },
    {
        "section": "Offshore Insurance Fund",
        "start_row": 73,
        "end_row": 127,
        "column_indices": list(range(2, 12)),
        "column_names": [
            DESCRIPTION_COLUMN,
            "Row No.",
            "Cargo",
            "Marine Hull",
            "Aviation Hull",
            "Property",
            "Motor",
            "Engineering",
            "Liability and Others",
            "Total",
        ],
    },
]


def extract_insurer_name(raw_sheet):
    """Read the insurer name from the top of the worksheet."""
    name_prefix = "NAME OF INSURER:"
    top_rows = raw_sheet.head(10)

    for value in top_rows.stack().dropna():
        text = str(value).strip()
        if text.upper().startswith(name_prefix):
            return text.split(":", 1)[1].strip()

    raise ValueError("Could not find insurer name in the top rows of the worksheet.")


def clean_section(raw_sheet, section, insurer_name):
    """Extract and clean one fund section from G1."""
    start_row = section["start_row"]
    end_row = section["end_row"]
    table = raw_sheet.iloc[start_row:end_row, section["column_indices"]].copy()
    table.columns = section["column_names"]

    # Keep only real data rows, excluding empty lines and repeated table headers.
    table = table.dropna(how="all")
    table = table[table[DESCRIPTION_COLUMN].notna()]
    table = table[table[DESCRIPTION_COLUMN] != DESCRIPTION_COLUMN]
    # Trim source indentation in the label column; keep pandas' nullable string dtype.
    table[DESCRIPTION_COLUMN] = table[DESCRIPTION_COLUMN].astype("string").str.strip()

    # Convert merged visual headings into category columns, then keep only data rows.
    value_columns = [col for col in table.columns if col not in {DESCRIPTION_COLUMN, "Row No."}]
    heading_rows = table["Row No."].isna() & table[value_columns].isna().all(axis=1)
    major_heading_rows = heading_rows & table[DESCRIPTION_COLUMN].str.fullmatch(
        r"[A-Z0-9/ ()'\-]+",
        na=False,
    )

    table["Category"] = table[DESCRIPTION_COLUMN].where(major_heading_rows).ffill()
    table["Subcategory"] = table[DESCRIPTION_COLUMN].where(
        heading_rows & ~major_heading_rows
    ).ffill()
    table = table[~heading_rows].copy()

    table.insert(0, "Section", section["section"])
    table.insert(0, "insurer", insurer_name)

    # Convert row numbers and operating values to numbers.
    numeric_columns = [
        col
        for col in table.columns
        if col not in {"insurer", "Section", "Category", "Subcategory", DESCRIPTION_COLUMN}
    ]
    table[numeric_columns] = table[numeric_columns].apply(pd.to_numeric, errors="coerce")

    return table.reindex(columns=G1_OUTPUT_COLUMNS).reset_index(drop=True)


def clean_g1_table(raw_sheet):
    """Clean both G1 fund sections and combine them into one table."""
    insurer_name = extract_insurer_name(raw_sheet)
    cleaned_sections = [
        clean_section(raw_sheet, section, insurer_name)
        for section in G1_TABLE_SECTIONS
    ]
    return pd.concat(cleaned_sections, ignore_index=True)


FUND_SHEET_SUFFIXES = {
    "Singapore Insurance Fund": "SIF",
    "Offshore Insurance Fund": "OIF",
}


def normalise_company_name(company_name):
    """Normalise company names so G1 form names can be matched to Mapping.xlsx."""
    text = str(company_name).replace("\xa0", " ").upper()
    text = text.replace("&", "AND")
    text = re.sub(r"\bLIMITED\b", "LTD", text)
    text = re.sub(r"\bASSN\b", "ASSOCIATION", text)
    return re.sub(r"[^A-Z0-9]", "", text)


def company_name_from_file_path(file_path):
    """Extract the company-name part from a MAS form file name."""
    stem = file_path.stem
    stem = re.sub(r"^(?:[IR][A-Z0-9]{3}|RA\d{2}|RB\d{2})G_", "", stem)
    stem = re.sub(r"_\d{4}.*$", "", stem)
    return stem.replace("_", " ")


def load_company_code_mapping(mapping_file):
    """Load company-name-to-code mapping from Knowledge/Mapping.xlsx."""
    mapping_raw = pd.read_excel(mapping_file, header=None)
    header_rows = mapping_raw.apply(
        lambda row: row.astype(str).str.contains("Code", case=False, na=False).any(),
        axis=1,
    )
    if not header_rows.any():
        raise ValueError(f"Could not find a Code header row in {mapping_file}")

    header_row = header_rows.idxmax()
    mapping = mapping_raw.iloc[header_row + 1:].copy()
    mapping.columns = mapping_raw.iloc[header_row].astype(str).str.strip()
    mapping = mapping.rename(columns=lambda col: str(col).strip())
    mapping = mapping.dropna(subset=["Code", "Company Name"])

    company_code_by_name = {}
    for _, row in mapping.iterrows():
        company_code = str(row["Code"]).strip()
        company_name = str(row["Company Name"]).strip()
        normalised_name = normalise_company_name(company_name)
        if normalised_name:
            company_code_by_name[normalised_name] = company_code

    return company_code_by_name


def get_company_code(insurer_name, company_code_by_name, file_path=None, fuzzy_threshold=0.86):
    """Return the mapped company code for an insurer name from the raw G1 form."""
    candidate_names = [insurer_name]
    if file_path is not None:
        candidate_names.append(company_name_from_file_path(file_path))

    normalised_candidates = [
        normalise_company_name(candidate)
        for candidate in candidate_names
        if str(candidate).strip()
    ]

    for normalised_name in normalised_candidates:
        company_code = company_code_by_name.get(normalised_name)
        if company_code:
            return company_code

    for normalised_name in normalised_candidates:
        containing_matches = [
            mapped_name
            for mapped_name in company_code_by_name
            if normalised_name in mapped_name or mapped_name in normalised_name
        ]
        if containing_matches:
            best_match = max(containing_matches, key=len)
            return company_code_by_name[best_match]

    best_score = 0
    best_match = None
    for normalised_name in normalised_candidates:
        for mapped_name in company_code_by_name:
            score = SequenceMatcher(None, normalised_name, mapped_name).ratio()
            if score > best_score:
                best_score = score
                best_match = mapped_name

    if best_match and best_score >= fuzzy_threshold:
        return company_code_by_name[best_match]

    raise KeyError(
        "Could not map insurer name to company code in Mapping.xlsx: "
        f"{insurer_name}"
    )


def make_excel_sheet_name(file_path, fund_name, company_code):
    """Create a valid Excel sheet name in <Company Code>_<Fund>_<Year> format."""
    invalid_characters = str.maketrans({char: "_" for char in "[]:*?/\\"})
    stem_parts = file_path.stem.rsplit("_", 1)

    if len(stem_parts) == 2 and stem_parts[1].isdigit():
        year = stem_parts[1]
    else:
        year = ""

    fund_suffix = FUND_SHEET_SUFFIXES[fund_name]
    sheet_suffix = f"_{fund_suffix}_{year}" if year else f"_{fund_suffix}"
    max_company_length = 31 - len(sheet_suffix)
    company_code = str(company_code).translate(invalid_characters)[:max_company_length]

    return f"{company_code}{sheet_suffix}"


def autofit_columns(worksheet, padding=2, max_width=80):
    """Resize worksheet columns based on the longest visible value in each column."""
    for column_cells in worksheet.columns:
        max_length = max(len(str(cell.value or "")) for cell in column_cells)
        adjusted_width = min(max_length + padding, max_width)
        worksheet.column_dimensions[column_cells[0].column_letter].width = adjusted_width


## 4. Clean each workbook and export the result

Each source workbook is read from worksheet `G1`. The cleaned table is split by Singapore Insurance Fund and Offshore Insurance Fund, then written to `Data/Output_MAS_G1.xlsx`, with one worksheet per company code and fund. Existing output content is replaced when the notebook runs, so the report-generation notebook always uses the latest cleaned MAS G1 data.


In [28]:
company_code_by_name = load_company_code_mapping(MAPPING_FILE)
cleaned_tables = {}

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", mode="w") as writer:
    for source_file in source_files:
        print(f"Cleaning {source_file.name}")

        raw_sheet = pd.read_excel(
            source_file,
            sheet_name=SHEET_NAME,
            header=None,
        )

        cleaned_table = clean_g1_table(raw_sheet)
        insurer_name = cleaned_table["insurer"].dropna().iloc[0]
        company_code = get_company_code(insurer_name, company_code_by_name, source_file)

        for fund_name in FUND_SHEET_SUFFIXES:
            fund_table = cleaned_table[cleaned_table["Section"] == fund_name].reset_index(drop=True)
            sheet_name = make_excel_sheet_name(source_file, fund_name, company_code)
            cleaned_tables[sheet_name] = fund_table
            fund_table.to_excel(writer, sheet_name=sheet_name, index=False)

    for worksheet in writer.sheets.values():
        autofit_columns(worksheet)

print(f"Saved {len(cleaned_tables)} cleaned table(s)")


Cleaning I400G_AUTO_&_GENERAL_INSURANCE_SINGAPORE_PTE._LIMITED_2022.xlsx


/Users/thuhoang/opt/anaconda3/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Cleaning I400G_AUTO_&_GENERAL_INSURANCE_SINGAPORE_PTE._LIMITED_2023.xlsx


/Users/thuhoang/opt/anaconda3/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Cleaning I400G_AUTO_&_GENERAL_INSURANCE_SINGAPORE_PTE._LIMITED_2024.xlsx


...

Cleaning RB02G_HELVETIA_SWISS_INSURANCE_COMPANY_LTD_SINGAPORE_BRANCH_2024.xlsx


/Users/thuhoang/opt/anaconda3/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Cleaning RB03G_XL_RE_EUROPE_SE_SINGAPORE_BRANCH_2024.xlsx


/Users/thuhoang/opt/anaconda3/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Saved 454 cleaned table(s)


## 5. Quick preview

Use this preview to confirm that the first cleaned table looks sensible before using the exported Excel file in the reporting workflow.


In [29]:
first_sheet_name = next(iter(cleaned_tables))
cleaned_tables[first_sheet_name].head()


,insurer,Section,Category,Subcategory,Description,Row No.,Cargo,Marine Hull,Aviation Hull,Property,...,Personal Accident,Health,Public Liability/ Product Liability,Surety,Engineering,Professional Indemnity,Credit/ Credit-related,Others,Liability and Others,Total
0,AUTO & GENERAL INSURANCE (SINGAPORE) PTE. LIMITED,Singapore Insurance Fund,PREMIUMS,Gross premiums,Direct business,1,0,0,0,0,...,124116.0,0.0,0.0,0.0,0,0.0,0.0,0.0,NaN,28315487.0
1,AUTO & GENERAL INSURANCE (SINGAPORE) PTE. LIMITED,Singapore Insurance Fund,PREMIUMS,Reinsurance business accepted from cedants in -,Singapore,2,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,NaN,0.0
2,AUTO & GENERAL INSURANCE (SINGAPORE) PTE. LIMITED,Singapore Insurance Fund,PREMIUMS,Reinsurance business accepted from cedants in -,Other ASEAN countries,3,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,NaN,0.0
3,AUTO & GENERAL INSURANCE (SINGAPORE) PTE. LIMITED,Singapore Insurance Fund,PREMIUMS,Reinsurance business accepted from cedants in -,Other countries,4,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,NaN,0.0
4,AUTO & GENERAL INSURANCE (SINGAPORE) PTE. LIMITED,Singapore Insurance Fund,PREMIUMS,Reinsurance business accepted from cedants in -,Total (2 to 4),5,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,NaN,0.0
